In [ ]:
# ─── 1. Dependencias e Importacoes (Colab / Local) ────────────────────────────
#!pip install -q ruptures scikit-learn openpyxl matplotlib pandas numpy scipy pyarrow joblib

import os, warnings, time, zipfile
from datetime import datetime
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from sklearn.preprocessing import StandardScaler
from scipy import stats
from joblib import Parallel, delayed
import ruptures as rpt

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

%matplotlib inline
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#B0B0B0'
plt.rcParams['axes.linewidth'] = 0.8

TEMPOS = {}
print(f"OK deps | IN_COLAB={IN_COLAB} | CPUs={os.cpu_count()}")

In [ ]:
# ─── 2. Configuracao de Hiperparametros ───────────────────────────────────────
INPUT_FILE = 'sre_seg_200m_2110_2026_analitica.xlsx'
SRE_REFERENCIA = '320EGO0130' # sempre incluido na calibracao funcional e nas figuras de auditoria

JUMP     = 1          # Hiperparametro fixo
MIN_SIZE = 3          # n. minimo de intervalos por segmento (3 x 200 m = 600 m, piso operacional)

# ── Funcional (inalterado da v6): PELT l2 sobre IGG + banda +-MARGEM_PEN + BIC ──
MODEL_FUNC    = 'l2'
ALPHA_KM_FUNC = 3.0 / 25.23   # substituido pela calibracao da Celula 5 se CALIBRAR_FUNC
MARGEM_PEN    = 0.5
N_CAND_PEN    = 9
CALIBRAR_FUNC   = True
N_MIN_CALIB     = 50      # SREs com >= N intervalos entram na calibracao funcional
N_SRES_CALIB    = 120     # amostra estratificada por comprimento (deterministica)
ALPHA_GRID_FUNC = np.round(np.arange(0.05, 0.40 + 1e-9, 0.0125), 4)
CRITERIO_ALPHA_FUNC = 'objetivo'   # 'objetivo' = minimo da soma de BIC | 'cotovelo' = joelho quebras/km x alpha

# ── Estrutural (novo, Biswas & Kuna 2022): PELT 'normal' sobre d0 bruto + CROPS + cotovelo ──
MODEL_ESTRUT    = 'normal'  # log-verossimilhanca Gaussiana: muda media E/OU variancia (Eqs. 7-10)
MODO_PENALIDADE = 'rede'    # 'rede' = um beta para toda a malha (cotovelo da curva da rede)
                            # 'sre'  = cotovelo de cada SRE (procedimento literal do artigo)
BETA_MIN         = None     # inicio da varredura de beta; None = ln(N mediano dos SREs com d0 e N >= N_MIN_CALIB)
                            # (no artigo: beta de 5 a 35.6 com N = 161 -> ln(161) = 5.1)
BETA_MAX_QUANTIL = 0.90     # (rede) fim da curva = quantil do beta que zera as quebras de cada SRE
BETA_FIXO        = None     # (rede) beta fixo no lugar do cotovelo (ex.: para o artigo); None = cotovelo
                            # O cotovelo depende da faixa varrida (BETA_MIN, BETA_MAX_QUANTIL); a referencia
                            # BIC 3 ln(N) (media + variancia + posicao por quebra) e impressa na Celula 5.
N_GRID_BETA      = 400      # pontos da curva beta x n. de quebras (o caminho CROPS e exato; a grade so desenha/acha o cotovelo)
CROPS_MAX_RUNS   = 400      # limite de execucoes do PELT por SRE no CROPS
PELT_EXATO       = True     # poda adiada: otimo global exato com min_size > 1 (a poda do ruptures pode perde-lo)
VAR_FLOOR_FRAC   = 0.10     # piso de variancia = (frac x desvio do d0 no SRE)^2 -> trechos de d0 repetido nao viram quebra
N_MIN_COTOVELO_SRE = 4      # (sre) minimo de solucoes no caminho CROPS para um cotovelo proprio; senao usa o beta da rede

FUNDIR_FUNC   = True   # funcional: funde secoes vizinhas sem diferenca de media de IGG (Welch)
FUNDIR_ESTRUT = False  # B&K nao fundem (o cotovelo ja escolhe o n. de quebras); True = Welch + Brown-Forsythe
FUNDIR_COMB   = True   # combinada: funde se nao ha diferenca em d0 (media e variancia) nem em IGG (media)
ALPHA_TESTE   = 0.05   # nivel de significancia dos testes de fusao e da auditoria

SUBSEGMENTAR_COMB = True   # combinada em 2 etapas: PELT normal (mesmo beta) dentro de cada base funcional
BUFFER_KM         = 0.4    # quebra funcional a menos de BUFFER_KM de uma estrutural e absorvida por ela
MIN_COMP_COMB_KM  = 0.6
CV_LIMITE         = 1.0

# ── Auditoria de residuos (B&K, secao "Test of Assumptions") ──
N_LAGS_ACF      = 20       # lags da ACF / Ljung-Box (limitado a n/4)
N_MIN_AUDITORIA = 8        # SREs com menos d0 medidos nao sao auditados

GERAR_GRAFICOS = True
DPI            = 110
N_MIN_GRAFICO  = 5        # SREs com menos intervalos nao geram grafico
N_JOBS         = max(1, min(6, (os.cpu_count() or 2) - 2))  # cada worker ~250 MB; -1 = todos (cuidado com RAM)
USE_FAST_PELT  = True     # False = funcional usa rpt.Pelt (lento, mesmo resultado)
USAR_CACHE     = True     # cache parquet da leitura agregada

TS = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_FILE = f'segmentacao_rede_v8_{TS}.xlsx'
OUTPUT_DIR  = f'graficos_rede_v8_{TS}'
ZIP_FILE    = f'pacote_segmentacao_v8_{TS}.zip'
os.makedirs(OUTPUT_DIR, exist_ok=True)

assert MODO_PENALIDADE in ('rede', 'sre')
print(f"Configuracao carregada. Min_size={MIN_SIZE} | Jump={JUMP} | estrutural={MODEL_ESTRUT} sobre d0 | "
      f"penalidade={MODO_PENALIDADE} | piso_var={VAR_FLOOR_FRAC}")

In [ ]:
# ─── 3. Leitura de Dados (com cache) e Calculo do ACA ─────────────────────────
t0 = time.time()
if not os.path.exists(INPUT_FILE) and IN_COLAB:
    print(f"Arquivo '{INPUT_FILE}' nao encontrado. Selecione o arquivo para upload:")
    uploaded = files.upload()
    if uploaded:
        INPUT_FILE = list(uploaded.keys())[0]

COLS_LEITURA = ['sre', 'sre_segmento', 'km_inicial', 'km_final', 'igg', 'iri_medio', 'd0']

def carregar_dados(path):
    df = pd.read_excel(path, usecols=lambda c: c in COLS_LEITURA)
    for c in ['km_inicial', 'km_final', 'igg', 'iri_medio', 'd0']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    has_seg = 'sre_segmento' in df.columns
    group_cols = ['sre', 'sre_segmento'] if has_seg else ['sre', 'km_inicial', 'km_final']
    agg = df.groupby(group_cols, as_index=False).agg(
        km_inicial=('km_inicial', 'min'), km_final=('km_final', 'max'),
        igg=('igg', 'mean'), iri=('iri_medio', 'mean'), d0=('d0', 'mean'))
    agg = agg.dropna(subset=['km_inicial', 'km_final']).sort_values(['sre', 'km_inicial']).reset_index(drop=True)

    agg['d0_obs'] = agg['d0'].notna()   # d0 medido (antes do preenchimento) -> usado na auditoria
    for c in ['igg', 'iri', 'd0']:
        agg[c] = agg.groupby('sre')[c].transform(lambda s: s.interpolate(limit_direction='both').fillna(s.median()))

    agg['km_centro_km'] = 0.5 * (agg['km_inicial'] + agg['km_final'])
    agg['aca'] = agg.groupby('sre')['d0'].transform(lambda s: (s - s.mean()).cumsum())
    agg['sre'] = agg['sre'].astype(str)
    return agg

def carregar_dados_cache(path):
    cache = os.path.splitext(path)[0] + '_agg_cache_v7.parquet'
    if USAR_CACHE and os.path.exists(cache) and os.path.getmtime(cache) >= os.path.getmtime(path):
        print(f"Lendo cache: {cache}")
        return pd.read_parquet(cache)
    agg = carregar_dados(path)
    if USAR_CACHE:
        agg.to_parquet(cache, index=False)
        print(f"Cache gravado: {cache}")
    return agg

df = carregar_dados_cache(INPUT_FILE)
sres = sorted(df['sre'].unique().tolist())

# arrays numpy por SRE (evita filtrar o DataFrame inteiro a cada SRE)
grupos = {
    sre: {
        'ini': g['km_inicial'].to_numpy(float), 'fim': g['km_final'].to_numpy(float),
        'centro': g['km_centro_km'].to_numpy(float), 'igg': g['igg'].to_numpy(float),
        'iri': g['iri'].to_numpy(float), 'd0': g['d0'].to_numpy(float), 'aca': g['aca'].to_numpy(float),
        'd0_obs': g['d0_obs'].to_numpy(bool),
    }
    for sre, g in df.groupby('sre', sort=True)
}
TEMPOS['leitura'] = time.time() - t0
sem_d0  = sum(not np.isfinite(a['d0']).any() for a in grupos.values())
sem_igg = sum(not np.isfinite(a['igg']).any() for a in grupos.values())
com_d0  = [a for a in grupos.values() if np.isfinite(a['d0']).all() and len(a['d0']) > 1]
rep_d0  = np.mean(np.concatenate([np.diff(a['d0']) == 0 for a in com_d0])) if com_d0 else 0.0
print(f"SREs: {len(sres)} | Total intervalos: {len(df)} | d0 medido: {df['d0_obs'].mean():.1%} | {TEMPOS['leitura']:.1f}s")
print(f"AVISO: {sem_d0} SREs sem nenhum d0 e {sem_igg} sem nenhum IGG (saem com 1 segmento e estatisticas vazias)")
print(f"d0 consecutivos identicos (preenchimento de lacunas): {rep_d0:.1%} -> tratados pelo piso de variancia")

In [ ]:
# ─── 4. Funcoes Auxiliares (PELT rapido normal/l2, CROPS, cotovelo, fusao, auditoria) ─
# ── PELT rapido: replica rpt.Pelt(model, min_size, jump).fit(x).predict(pen) p/ sinal 1-D ──
# 'normal' = CostNormal do ruptures: custo do segmento = n * log(var), var com ddof=0, ou seja
# -2 x log-verossimilhanca Gaussiana com media E variancia proprias (Biswas & Kuna, 2022, Eqs. 7-10).
# O ruptures soma 1e-6 a var (piso dependente da unidade); aqui o piso e var_floor, relativo ao SRE.
# Um piso aditivo preserva a desigualdade C(a,b) + C(b,c) <= C(a,c), logo a poda do PELT continua exata.
_MIN_SIZE_CUSTO = {'l2': 1, 'normal': 2}  # cost.min_size interno do ruptures 1.1.x

def prefixos_pelt(y):
    y = np.asarray(y, float).ravel()
    y = y - y.mean()                 # custos l2/normal invariantes a translacao; reduz cancelamento numerico
    z = np.zeros(1)
    return np.concatenate([z, np.cumsum(y)]), np.concatenate([z, np.cumsum(y * y)])

def custos_segmentos(model, ts, e, P, var_floor=0.0):
    """Custo do segmento [t, e) para um vetor de inicios ts (mesmas formulas de CostL2 / CostNormal)."""
    S1, S2 = P
    n = (e - ts).astype(float)
    s1 = S1[e] - S1[ts]
    sq = S2[e] - S2[ts]
    if model == 'l2':
        return sq - s1 * s1 / n
    var = np.maximum(sq / n - (s1 / n) ** 2, 0.0)
    return n * np.log(var + var_floor)

def pelt_fast(signal, model, min_size, jump, pen, P=None, var_floor=0.0, exato=False):
    """PELT com custo O(1) por segmento; retorna bkps no formato do ruptures (inclui n).
    exato=False: poda identica ao rpt.Pelt (com min_size > 1 ela pode descartar o otimo em casos raros).
    exato=True : poda adiada -- t podado em s so sai a partir de s + min_size, quando (s, T) ja e um
                 ultimo segmento admissivel; recupera o otimo global exato (Killick et al., 2012)."""
    y = np.asarray(signal, float).ravel()
    n = len(y)
    min_size = max(int(min_size), _MIN_SIZE_CUSTO[model])
    if P is None:
        P = prefixos_pelt(y)
    F = np.full(n + 1, np.inf); F[0] = 0.0
    prev = np.zeros(n + 1, dtype=int)
    valido = np.zeros(n + 1, dtype=bool); valido[0] = True
    sai_em = np.full(n + 1, n + 1)               # (exato) instante a partir do qual o candidato e podado
    adm = np.empty(0, dtype=int)
    ind = [k for k in range(0, n, jump) if k >= min_size] + [n]
    for bkp in ind:
        novo = (bkp - min_size) // jump * jump
        if novo >= 0:
            adm = np.append(adm, novo)
        ts = adm[valido[adm] & (sai_em[adm] > bkp)]
        if len(ts) == 0:
            raise ValueError('sem particao admissivel')
        tot = F[ts] + (custos_segmentos(model, ts, bkp, P, var_floor) + pen)
        j = int(np.argmin(tot))
        F[bkp] = tot[j]; prev[bkp] = ts[j]; valido[bkp] = True
        if exato:
            podar = ts[tot > F[bkp] + pen]
            sai_em[podar] = np.minimum(sai_em[podar], bkp + min_size)
            adm = ts
        else:
            adm = ts[tot <= F[bkp] + pen]
    bkps, e = [], n
    while e > 0:
        bkps.append(int(e)); e = prev[e]
    return bkps[::-1]

def custo_total(model, bkps, P, var_floor=0.0):
    """Custo nao penalizado Q da segmentacao (soma dos custos dos segmentos)."""
    ini = np.array([0] + list(bkps[:-1])); fim = np.array(bkps)
    return float(sum(custos_segmentos(model, ini[i:i + 1], fim[i], P, var_floor)[0] for i in range(len(fim))))

def piso_variancia(d0):
    s = float(np.nanstd(d0)) if np.isfinite(d0).any() else 0.0
    return max((VAR_FLOOR_FRAC * s) ** 2, 1e-9)

# ── CROPS (Haynes, Eckley & Fearnhead, 2017): caminho exato de penalidades ──
def crops_normal(y, beta_min, min_size=MIN_SIZE, jump=JUMP, var_floor=1e-9, max_runs=CROPS_MAX_RUNS):
    """Todas as segmentacoes otimas do PELT 'normal' para beta em [beta_min, inf).
    Retorna {K: (Q, bkps)} so com os vertices da envoltoria convexa inferior de (K, Q), que sao
    exatamente as solucoes otimas para algum beta: K* (beta) = argmin_K Q_K + beta * K."""
    y = np.asarray(y, float).ravel()
    N = len(y)
    P = prefixos_pelt(y)
    sols = {0: (custo_total('normal', [N], P, var_floor), [N])}   # 0 quebras e sempre admissivel
    if N < 2 * max(min_size, _MIN_SIZE_CUSTO['normal']):
        return sols

    def run(beta):
        bk = pelt_fast(y, 'normal', min_size, jump, float(beta), P, var_floor, exato=PELT_EXATO)
        K, Q = len(bk) - 1, custo_total('normal', bk, P, var_floor)
        if K not in sols or Q < sols[K][0] - 1e-9:
            sols[K] = (Q, bk)
        return K, sols[K][0]

    K_lo, Q_lo = run(beta_min)
    b_hi = max(2.0 * beta_min, 1.0)
    K_hi, Q_hi = run(b_hi)
    n_runs = 2
    while K_hi > 0 and n_runs < max_runs:
        b_hi *= 2.0
        K_hi, Q_hi = run(b_hi); n_runs += 1
    pilha = [(beta_min, K_lo, Q_lo, b_hi, K_hi, Q_hi)]
    while pilha and n_runs < max_runs:
        b0, K0, Q0, b1, K1, Q1 = pilha.pop()
        if K0 <= K1 + 1:
            continue
        bi = (Q1 - Q0) / (K0 - K1)          # beta em que as duas solucoes empatam
        if not (b0 < bi < b1):
            continue
        Ki, Qi = run(bi); n_runs += 1
        if Ki not in (K0, K1):              # existe solucao intermediaria -> subdivide
            pilha += [(b0, K0, Q0, bi, Ki, Qi), (bi, Ki, Qi, b1, K1, Q1)]

    hull = []                               # envoltoria convexa inferior (monotone chain)
    for K in sorted(sols):
        while len(hull) >= 2:
            (k1, q1), (k2, q2) = hull[-2], hull[-1]
            if (k2 - k1) * (sols[K][0] - q1) - (q2 - q1) * (K - k1) <= 0:
                hull.pop()
            else:
                break
        hull.append((K, sols[K][0]))
    return {K: sols[K] for K, _ in hull}

def tabela_caminho(path, beta_min):
    """K, Q e intervalo [beta_inf, beta_sup] em que cada segmentacao do caminho e otima."""
    Ks = np.array(sorted(path), dtype=int)
    Q = np.array([path[k][0] for k in Ks])
    sup = np.r_[np.inf, (Q[:-1] - Q[1:]) / (Ks[1:] - Ks[:-1])]
    inf_ = np.r_[sup[1:], beta_min]
    return Ks, Q, inf_, sup

def k_otimo(Ks, Q, betas):
    betas = np.atleast_1d(np.asarray(betas, float))
    return Ks[np.argmin(Q[None, :] + betas[:, None] * Ks[None, :], axis=1)]

def kneedle_idx(x, y):
    """Cotovelo da curva (maior distancia a corda entre extremos, eixos normalizados)."""
    x = np.asarray(x, float); y = np.asarray(y, float)
    if len(x) < 3 or np.ptp(y) == 0:
        return 0
    xn = (x - x.min()) / max(np.ptp(x), 1e-12)
    yn = (y - y.min()) / max(np.ptp(y), 1e-12)
    dist = np.abs((yn[-1] - yn[0]) * xn - (xn[-1] - xn[0]) * yn + xn[-1] * yn[0] - yn[-1] * xn[0])
    return int(np.argmax(dist))

def kneedle(x, y):
    return float(np.asarray(x, float)[kneedle_idx(x, y)])

def cotovelo_sre(Ks, Q, inf_, sup, beta_min, n_grid=N_GRID_BETA):
    """Grafico de diagnostico do SRE (B&K, Figs. 8-9): n. de quebras x beta de beta_min ate o beta que
    zera as quebras; o cotovelo define K*, e o beta escolhido e o centro do intervalo em que K* e otimo."""
    if len(Ks) < N_MIN_COTOVELO_SRE:
        return None
    grid = np.linspace(beta_min, sup[1], n_grid)
    K_est = int(k_otimo(Ks, Q, [kneedle(grid, k_otimo(Ks, Q, grid))])[0])
    i = int(np.flatnonzero(Ks == K_est)[0])
    beta_sel = 0.5 * (inf_[i] + sup[i]) if np.isfinite(sup[i]) else sup[1] * 1.05
    return K_est, float(beta_sel)

# ── Funcional (igual a v6) ──
def banda_penalidades(base_pen, margem=MARGEM_PEN, n=N_CAND_PEN):
    base_pen = float(base_pen)
    if base_pen <= 0 or margem <= 0 or n <= 1:
        return [max(base_pen, 0.05)]
    lo = max(0.05, base_pen * (1.0 - margem))
    hi = base_pen * (1.0 + margem)
    return [float(v) for v in np.linspace(lo, hi, int(n))]

def pelt_breaks_multipen(Xs, modelo, min_size, jump, pens):
    Xs = np.asarray(Xs, float)
    if USE_FAST_PELT and modelo in _MIN_SIZE_CUSTO and Xs.reshape(len(Xs), -1).shape[1] == 1:
        y = Xs.ravel()
        P = prefixos_pelt(y)
        out = {}
        for pen in pens:
            try:
                bkps = pelt_fast(y, modelo, min_size, jump, float(pen), P)
                out[float(pen)] = [int(b) - 1 for b in bkps[:-1]]
            except Exception:
                out[float(pen)] = []
        return out
    try:
        algo = rpt.Pelt(model=modelo, min_size=min_size, jump=jump).fit(Xs)
    except Exception:
        return {float(p): [] for p in pens}
    out = {}
    for pen in pens:
        try:
            bkps = algo.predict(pen=float(pen))
            out[float(pen)] = [int(b) - 1 for b in bkps[:-1]]
        except Exception:
            out[float(pen)] = []
    return out

def labels_from_bkps(bkps, N):
    labels = np.zeros(N, dtype=int)
    prev = 0
    for s_idx, bp in enumerate(bkps):
        labels[prev:int(bp)] = s_idx
        prev = int(bp)
    return labels

def bic_l2(X, labels):
    N = len(X)
    D = X.shape[1]
    sse = 0.0
    for lbl in np.unique(labels):
        pts = X[labels == lbl]
        sse += float(((pts - pts.mean(axis=0)) ** 2).sum())
    k = len(np.unique(labels)) * D + (len(np.unique(labels)) - 1)
    return N * np.log(max(sse, 1e-12) / N) + k * np.log(N)

def media_cv(v):
    if len(v) == 0:
        return None, None
    m = float(np.mean(v))
    s = float(np.std(v)) if len(v) > 1 else 0.0
    cv = s / abs(m) if abs(m) > 1e-9 else np.nan
    return m, cv

# ── Fusao de secoes vizinhas (CDA modificado, B&K Fig. 3) ──
def _p_welch(x, y):
    """Diferenca de media (Welch)."""
    if len(x) < 2 or len(y) < 2:
        return 1.0                      # amostra pequena demais para provar diferenca
    with warnings.catch_warnings():     # workers do joblib nao herdam o filtro da Celula 1
        warnings.simplefilter('ignore')
        p = stats.ttest_ind(x, y, equal_var=False).pvalue
    if np.isnan(p):                     # variancias nulas
        return 1.0 if np.isclose(np.mean(x), np.mean(y)) else 0.0
    return float(p)

def _p_bf(x, y):
    """Diferenca de variancia (Brown-Forsythe = Levene com mediana; robusto a nao-normalidade)."""
    if len(x) < 2 or len(y) < 2:
        return 1.0
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        p = stats.levene(x, y, center='median').pvalue
    if np.isnan(p):
        return 1.0 if np.isclose(np.var(x), np.var(y)) else 0.0
    return float(p)

def fundir_testes(inicios, series, N, alpha=ALPHA_TESTE):
    """Funde secoes consecutivas sem diferenca significativa em NENHUMA das series.
    `series` = [(array, testa_variancia)]; com testa_variancia=True a quebra so cai se nem a media (Welch)
    nem a variancia (Brown-Forsythe) diferem, preservando as quebras de variancia do custo 'normal'.
    `inicios` = indices do primeiro intervalo de cada segmento (exceto o 0). Funde primeiro o par mais parecido."""
    fr = sorted(int(s) for s in inicios)
    while fr:
        lim = [0] + fr + [N]
        ps = []
        for i in range(len(fr)):
            p = 1.0
            for s, testa_var in series:
                x, y = s[lim[i]:lim[i + 1]], s[lim[i + 1]:lim[i + 2]]
                p = min(p, _p_welch(x, y), _p_bf(x, y) if testa_var else 1.0)
            ps.append(p)
        j = int(np.argmax(ps))
        if ps[j] <= alpha:
            break
        fr.pop(j)
    return fr

# ── Segmentacoes (indices: cada quebra = indice do 1o intervalo do segmento seguinte) ──
def _extensao(a, lo, hi):
    fim = a['ini'][hi] if hi < len(a['ini']) else float(np.max(a['fim'][lo:hi]))
    return float(fim - np.min(a['ini'][lo:hi]))

def segmentar_funcional_sre(a, alpha):
    igg = a['igg']
    N = len(igg)
    pens = banda_penalidades(alpha * _extensao(a, 0, N), MARGEM_PEN, N_CAND_PEN)
    X = np.asarray(igg, float).reshape(-1, 1)
    Xs = StandardScaler().fit_transform(X) if N > 1 else X
    breaks_map = pelt_breaks_multipen(Xs, MODEL_FUNC, MIN_SIZE, JUMP, pens)
    melhor = None
    for pen in pens:
        breaks = breaks_map[pen]
        labels = labels_from_bkps([b + 1 for b in breaks] + [N], N)
        bic = bic_l2(Xs, labels)
        if melhor is None or bic < melhor[0]:
            melhor = (bic, pen, breaks)
    bic, pen, breaks = melhor
    return [b + 1 for b in breaks], pen, bic

def segmentar_estrutural_trecho(d0, lo, hi, beta, var_floor):
    """PELT 'normal' sobre d0 bruto em [lo, hi) com penalidade beta (sem padronizar, sem snap)."""
    if hi - lo < 2 * MIN_SIZE:
        return []
    bk = pelt_fast(d0[lo:hi], MODEL_ESTRUT, MIN_SIZE, JUMP, float(beta), var_floor=var_floor, exato=PELT_EXATO)
    return [lo + b for b in bk[:-1]]

def reconciliar_combinada(func_km, sub_km_raw, km_min, km_max, min_comp_km=MIN_COMP_COMB_KM, buffer_km=BUFFER_KM):
    fronteiras = list(sub_km_raw)
    for fk in func_km:
        if not any(abs(fk - sk) <= buffer_km for sk in sub_km_raw):
            fronteiras.append(fk)
    fronteiras = sorted(set(fronteiras))
    limpas = []
    ultimo = km_min
    for f in fronteiras:
        if (f - ultimo) >= min_comp_km and (km_max - f) >= min_comp_km:
            limpas.append(f)
            ultimo = f
    return limpas

# ── Auditoria de residuos (B&K, "Test of Assumptions": Tab. 1, Figs. 11-13) ──
def residuos_segmentos(y, obs, inicios):
    """Residuo bruto r = d0 - media do segmento e padronizado z = r / desvio do segmento (ddof=1),
    so nos d0 medidos. O modelo 'normal' admite variancia propria por segmento, entao a normalidade
    e testada em z; r (como no artigo) mistura variancias e tende a caudas pesadas."""
    N = len(y)
    lim = [0] + sorted(int(s) for s in inicios) + [N]
    r = np.full(N, np.nan); z = np.full(N, np.nan); grupos_r, n_sd = [], []
    for i in range(len(lim) - 1):
        idx = np.arange(lim[i], lim[i + 1])[obs[lim[i]:lim[i + 1]]]
        if len(idx) == 0:
            continue
        v = y[idx]
        r[idx] = v - v.mean()
        if len(idx) > 1:
            sd = float(v.std(ddof=1))
            n_sd.append((len(idx), sd))
            grupos_r.append(r[idx])
            if sd > 0:
                z[idx] = r[idx] / sd
    return r, z, grupos_r, n_sd

def auditar_residuos(y, obs, inicios, n_lags=N_LAGS_ACF, alpha=ALPHA_TESTE):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        return _auditar_residuos(y, obs, inicios, n_lags, alpha)

def _auditar_residuos(y, obs, inicios, n_lags, alpha):
    r, z, grupos_r, n_sd = residuos_segmentos(y, obs, inicios)
    rr, zz = r[np.isfinite(r)], z[np.isfinite(z)]
    out = {'n_residuos': int(len(rr)), 'n_segmentos': len(inicios) + 1}
    if len(rr) < N_MIN_AUDITORIA or len(zz) < 3 or np.ptp(zz) == 0:
        return out, z
    sw_W, sw_p = stats.shapiro(zz) if len(zz) <= 5000 else (np.nan, np.nan)
    ks_D, ks_p = stats.kstest(zz, 'norm')
    sw_p_bruto = stats.shapiro(rr).pvalue if 3 <= len(rr) <= 5000 and np.ptp(rr) > 0 else np.nan
    # ACF dos residuos ao longo do km (B&K Fig. 12) + Ljung-Box
    n = len(rr)
    h = int(max(1, min(n_lags, n // 4)))
    rc = rr - rr.mean(); den = float((rc ** 2).sum())
    acf = np.array([(rc[:-k] * rc[k:]).sum() / den for k in range(1, h + 1)]) if den > 0 else np.zeros(h)
    banda = 1.96 / np.sqrt(n)
    lb_Q = n * (n + 2) * float(np.sum(acf ** 2 / (n - np.arange(1, h + 1))))
    lb_p = float(stats.chi2.sf(lb_Q, h))
    # heterocedasticidade entre segmentos (B&K Fig. 13) -> justifica 'normal' em vez de 'l2'
    validos = [g for g in grupos_r if len(g) >= 2]
    bf_p = float(stats.levene(*validos, center='median').pvalue) if len(validos) >= 2 else np.nan
    sds = np.array([s for _, s in n_sd if s > 0])
    out.update({
        'shapiro_W': round(float(sw_W), 4), 'shapiro_p': round(float(sw_p), 4),
        'ks_D': round(float(ks_D), 4), 'ks_p': round(float(ks_p), 4),
        'shapiro_p_res_bruto': round(float(sw_p_bruto), 4),
        'normalidade_ok': bool(sw_p > alpha and ks_p > alpha),
        'acf_lag1': round(float(acf[0]), 4), 'acf_max_abs': round(float(np.max(np.abs(acf))), 4),
        'banda_acf_95': round(float(banda), 4), 'lags_fora_banda': int(np.sum(np.abs(acf) > banda)), 'n_lags': h,
        'ljung_box_Q': round(lb_Q, 3), 'ljung_box_p': round(lb_p, 4), 'independencia_ok': bool(lb_p > alpha),
        'brown_forsythe_p': round(bf_p, 4) if np.isfinite(bf_p) else np.nan,
        'heterocedastico': bool(np.isfinite(bf_p) and bf_p <= alpha),
        'razao_desvio_max_min': round(float(sds.max() / sds.min()), 3) if len(sds) >= 2 else np.nan,
        'erro_padrao_medio': round(float(np.mean([s / np.sqrt(k) for k, s in n_sd])), 3) if n_sd else np.nan,
    })
    return out, z

print("OK helpers")

In [ ]:
# ─── 5. Calibracao: alpha funcional (BIC) + caminhos CROPS e cotovelo estrutural ──
# ── A) Funcional: grade de alpha_km avaliada nos SREs de calibracao (igual a v6) ──
def _calib_func_sre(a, grid_f):
    N = len(a['ini'])
    ext = max(_extensao(a, 0, N), 1e-9)
    bic, nq = [], []
    for al in grid_f:
        fr, _, b = segmentar_funcional_sre(a, al)
        bic.append(b); nq.append(len(fr) / ext)
    return np.array(bic), np.array(nq)

def escolher_sres_calibracao(grupos):
    # so entram SREs com IGG medido (sem dados -> objetivo constante, so dilui a soma)
    tam = pd.Series({s: len(a['ini']) for s, a in grupos.items() if np.isfinite(a['igg']).all()})
    cand = tam[tam >= N_MIN_CALIB].sort_values(kind='stable')
    if cand.empty:
        cand = tam.sort_values(kind='stable').tail(10)
    if len(cand) > N_SRES_CALIB:   # amostra estratificada por comprimento (sempre a mesma)
        cand = cand.iloc[np.unique(np.linspace(0, len(cand) - 1, N_SRES_CALIB).round().astype(int))]
    sel = list(cand.index)
    if SRE_REFERENCIA in grupos and SRE_REFERENCIA not in sel:
        sel.append(SRE_REFERENCIA)
    return sel

def calibrar_funcional(grupos):
    sel = escolher_sres_calibracao(grupos)
    res = Parallel(n_jobs=N_JOBS)(delayed(_calib_func_sre)(grupos[s], ALPHA_GRID_FUNC) for s in sel)
    obj = np.nansum(np.vstack([r[0] for r in res]), axis=0)   # verossimilhanca conjunta (min)
    nq = np.vstack([r[1] for r in res]).mean(axis=0)
    a_obj = float(ALPHA_GRID_FUNC[int(np.argmin(obj))])
    a_cot = kneedle(ALPHA_GRID_FUNC, nq)
    if a_obj in (float(ALPHA_GRID_FUNC[0]), float(ALPHA_GRID_FUNC[-1])):
        print(f"  AVISO: otimo do BIC funcional na borda da grade ({a_obj:.4f}) -> prefira 'cotovelo'")
    best = a_cot if CRITERIO_ALPHA_FUNC == 'cotovelo' else a_obj
    curva = pd.DataFrame({'alpha_km': ALPHA_GRID_FUNC, 'soma_bic': obj, 'quebras_por_km': nq})
    print(f"  Funcional ({len(sel)} SREs): objetivo={a_obj:.4f} | cotovelo={a_cot:.4f} -> "
          f"ALPHA_KM_FUNC={best:.4f} ({CRITERIO_ALPHA_FUNC})")
    return best, curva, sel, (nq, obj, a_obj, a_cot)

# ── B) Estrutural: caminho CROPS exato de cada SRE (PELT 'normal' sobre d0 bruto) ──
def caminho_sre(a, beta_min):
    d0 = a['d0']
    path = crops_normal(d0, beta_min, MIN_SIZE, JUMP, piso_variancia(d0))
    Ks, Q, inf_, sup = tabela_caminho(path, beta_min)
    return {'Ks': Ks, 'Q': Q, 'inf': inf_, 'sup': sup, 'bkps': {int(k): path[k][1] for k in Ks}}

t0 = time.time()
if CALIBRAR_FUNC:
    ALPHA_KM_FUNC_OPT, CURVA_FUNC, SRES_CALIB, _cf = calibrar_funcional(grupos)
else:
    ALPHA_KM_FUNC_OPT, CURVA_FUNC, SRES_CALIB, _cf = ALPHA_KM_FUNC, None, [], None
TEMPOS['calib_funcional'] = time.time() - t0

t0 = time.time()
N_D0 = pd.Series({s: len(a['d0']) for s, a in grupos.items() if np.isfinite(a['d0']).all()}, dtype=float)
if BETA_MIN is None:
    base = N_D0[N_D0 >= N_MIN_CALIB] if (N_D0 >= N_MIN_CALIB).any() else N_D0
    BETA_MIN_EF = float(np.log(np.median(base)))
else:
    BETA_MIN_EF = float(BETA_MIN)
_base_bic = N_D0[N_D0 >= N_MIN_CALIB] if (N_D0 >= N_MIN_CALIB).any() else N_D0
BETA_BIC = float(3 * np.log(np.median(_base_bic)))   # referencia: BIC com 3 parametros por quebra (mu, sigma, posicao)
SRES_D0 = [s for s in sres if s in N_D0.index and N_D0[s] >= 2 * MIN_SIZE]
_cams = Parallel(n_jobs=N_JOBS, batch_size=8)(delayed(caminho_sre)(grupos[s], BETA_MIN_EF) for s in SRES_D0)
CAMINHOS = dict(zip(SRES_D0, _cams))
EXT = {s: _extensao(grupos[s], 0, len(grupos[s]['ini'])) for s in sres}

# ── C) Grafico de diagnostico da REDE: beta x total de quebras (B&K Fig. 9, somado nos SREs) ──
BETA_ZERO = pd.Series({s: c['sup'][1] for s, c in CAMINHOS.items() if len(c['Ks']) > 1})
BETA_MAX_REDE = float(BETA_ZERO.quantile(BETA_MAX_QUANTIL)) if len(BETA_ZERO) else 10 * BETA_MIN_EF
GRID_BETA = np.linspace(BETA_MIN_EF, BETA_MAX_REDE, N_GRID_BETA)
K_REDE = np.zeros(len(GRID_BETA))
for c in CAMINHOS.values():
    K_REDE += k_otimo(c['Ks'], c['Q'], GRID_BETA)
EXT_D0 = sum(EXT[s] for s in CAMINHOS)
BETA_COTOVELO_REDE = kneedle(GRID_BETA, K_REDE)
BETA_REDE = float(BETA_FIXO) if BETA_FIXO is not None else BETA_COTOVELO_REDE

# ── D) Cotovelo de cada SRE (procedimento literal do artigo) e beta usado ──
COTOVELO = {s: cotovelo_sre(c['Ks'], c['Q'], c['inf'], c['sup'], BETA_MIN_EF) for s, c in CAMINHOS.items()}
BETA_SRE, ORIGEM_BETA = {}, {}
for s in CAMINHOS:
    if MODO_PENALIDADE == 'sre' and COTOVELO[s] is not None:
        BETA_SRE[s], ORIGEM_BETA[s] = COTOVELO[s][1], 'cotovelo_sre'
    else:
        BETA_SRE[s], ORIGEM_BETA[s] = BETA_REDE, ('rede' if MODO_PENALIDADE == 'rede' else 'fallback_rede')
TEMPOS['calib_estrutural'] = time.time() - t0

DIAG_REDE = pd.DataFrame({'beta': GRID_BETA, 'quebras_rede': K_REDE.astype(int),
                          'quebras_por_km': K_REDE / max(EXT_D0, 1e-9)})
TAB_CROPS = pd.DataFrame([
    {'sre': s, 'K_quebras': int(k), 'custo_Q': round(float(q), 4), 'beta_inf': round(float(bi), 4),
     'beta_sup': (round(float(bs), 4) if np.isfinite(bs) else None),
     'cotovelo_sre': bool(COTOVELO[s] is not None and COTOVELO[s][0] == k),
     'escolhido': bool(k == k_otimo(c['Ks'], c['Q'], [BETA_SRE[s]])[0])}
    for s, c in CAMINHOS.items() for k, q, bi, bs in zip(c['Ks'], c['Q'], c['inf'], c['sup'])])

b_cot = np.array([v[1] for v in COTOVELO.values() if v is not None])
k_rede_sel = int(K_REDE[np.argmin(np.abs(GRID_BETA - BETA_REDE))])
print(f"  Estrutural: CROPS em {len(CAMINHOS)} SREs com d0 ({TAB_CROPS.shape[0]} solucoes otimas) | "
      f"beta de {BETA_MIN_EF:.2f} a {BETA_MAX_REDE:.2f} (q{int(BETA_MAX_QUANTIL * 100)} do beta que zera as quebras)")
print(f"  Cotovelo da rede: beta={BETA_COTOVELO_REDE:.2f} | referencia BIC 3 ln(N)={BETA_BIC:.2f}"
      + (f" | BETA_FIXO={BETA_FIXO}" if BETA_FIXO is not None else ''))
print(f"  BETA_REDE={BETA_REDE:.2f} -> {k_rede_sel} quebras na malha ({k_rede_sel / max(EXT_D0, 1e-9):.3f}/km)")
if len(b_cot):
    print(f"  Cotovelo por SRE ({len(b_cot)} SREs com >= {N_MIN_COTOVELO_SRE} solucoes): beta mediano={np.median(b_cot):.2f} "
          f"(p25={np.percentile(b_cot, 25):.2f}, p75={np.percentile(b_cot, 75):.2f})")
print(f"✓ Usando: ALPHA_KM_FUNC={ALPHA_KM_FUNC_OPT:.4f} | estrutural MODO_PENALIDADE='{MODO_PENALIDADE}'"
      + (f" (beta={BETA_REDE:.2f})" if MODO_PENALIDADE == 'rede' else
         f" ({sum(v == 'cotovelo_sre' for v in ORIGEM_BETA.values())} cotovelos proprios, "
         f"{sum(v == 'fallback_rede' for v in ORIGEM_BETA.values())} com beta da rede)"))

# ── Figura de calibracao ──
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
ax = axes[0, 0]
if _cf is not None:
    nq, obj, a_obj, a_cot = _cf
    ax.plot(ALPHA_GRID_FUNC, nq, 'o-', color='#1F4E79', ms=3, label='quebras/km (media)')
    ax2 = ax.twinx(); ax2.plot(ALPHA_GRID_FUNC, obj, color='#C65A57', lw=1.2); ax2.set_ylabel('soma BIC', color='#C65A57')
    ax.axvline(a_obj, color='#C65A57', lw=1.2, label=f'BIC min: {a_obj:.4f}')
    ax.axvline(a_cot, color='#555555', ls='--', lw=1.2, label=f'cotovelo: {a_cot:.4f}')
    ax.legend(fontsize=8, loc='upper right')
ax.set_xlabel('alpha_km (penalidade por km)'); ax.set_ylabel('quebras por km', color='#1F4E79')
ax.set_title('(a) Funcional: IGG l2 - calibracao de alpha_km')

ax = axes[0, 1]
ax.step(GRID_BETA, K_REDE, where='post', color='#1F4E79', lw=1.3)
ax.axvline(BETA_COTOVELO_REDE, color='#C65A57', lw=1.3, label=f'cotovelo rede: beta={BETA_COTOVELO_REDE:.2f}')
ax.axvline(BETA_BIC, color='#2E7D32', ls=':', lw=1.3, label=f'BIC 3 ln(N): {BETA_BIC:.2f}')
if BETA_FIXO is not None:
    ax.axvline(BETA_REDE, color='#555555', ls='--', lw=1.3, label=f'BETA_FIXO: {BETA_REDE:.2f}')
ax.set_xlabel('penalidade beta'); ax.set_ylabel('n. de quebras na malha')
ax.set_title('(b) Estrutural: diagnostico da rede (d0, PELT normal, CROPS)')
ax.legend(fontsize=8)

ax = axes[1, 0]
ref = SRE_REFERENCIA if SRE_REFERENCIA in CAMINHOS else (max(CAMINHOS, key=lambda s: len(grupos[s]['ini'])) if CAMINHOS else None)
if ref is not None:
    c = CAMINHOS[ref]
    g = np.linspace(BETA_MIN_EF, max(c['sup'][1] * 1.1, BETA_MIN_EF + 1), N_GRID_BETA) if len(c['Ks']) > 1 else GRID_BETA
    ax.step(g, k_otimo(c['Ks'], c['Q'], g), where='post', color='#1F4E79', lw=1.3)
    ax.plot(c['inf'][1:], c['Ks'][1:], 'o', color='#1F4E79', ms=3)
    if COTOVELO.get(ref) is not None:
        ax.axvline(COTOVELO[ref][1], color='#555555', ls='--', lw=1.2,
                   label=f'cotovelo SRE: beta={COTOVELO[ref][1]:.2f} (K={COTOVELO[ref][0]})')
    ax.axvline(BETA_REDE, color='#C65A57', lw=1.2, label=f'beta rede: {BETA_REDE:.2f}')
    ax.set_title(f'(c) Diagnostico do SRE {ref} (B&K Fig. 9)'); ax.legend(fontsize=8)
ax.set_xlabel('penalidade beta'); ax.set_ylabel('n. de quebras')

ax = axes[1, 1]
if len(b_cot):
    ax.hist(np.clip(b_cot, None, BETA_MAX_REDE * 1.5), bins=40, color='#9DB4CC', edgecolor='white')
    ax.axvline(BETA_REDE, color='#C65A57', lw=1.3, label=f'beta rede: {BETA_REDE:.2f}')
    ax.axvline(np.median(b_cot), color='#555555', ls='--', lw=1.2, label=f'mediana SREs: {np.median(b_cot):.2f}')
    ax.legend(fontsize=8)
ax.set_xlabel('beta do cotovelo de cada SRE'); ax.set_ylabel('n. de SREs')
ax.set_title('(d) Dispersao do cotovelo individual')
for ax in axes.ravel():
    ax.grid(True, ls=':', alpha=0.5)
plt.tight_layout()
CALIB_PNG = os.path.join(OUTPUT_DIR, '_calibracao.png')
plt.savefig(CALIB_PNG, dpi=DPI, bbox_inches='tight'); plt.show()
print(f"Calibracao: funcional {TEMPOS['calib_funcional']:.1f}s | estrutural {TEMPOS['calib_estrutural']:.1f}s")

In [ ]:
# ─── 6. Execucao das 3 Segmentacoes por SRE (paralelo) ────────────────────────
def atributos(a, lo, hi, km_a, km_b):
    d0 = a['d0'][lo:hi]
    igg = a['igg'][lo:hi]
    iri = a['iri'][lo:hi]
    d0m, d0cv   = media_cv(d0)
    iggm, iggcv = media_cv(igg)
    irim, iricv = media_cv(iri)
    cvs = [c for c in (d0cv, iggcv, iricv) if c is not None and not (isinstance(c, float) and np.isnan(c))]
    coerente = bool(cvs) and max(cvs) <= CV_LIMITE
    d0sd  = float(np.std(d0, ddof=1)) if len(d0) > 1 and np.isfinite(d0).all() else None
    iggsd = float(np.std(igg, ddof=1)) if len(igg) > 1 and np.isfinite(igg).all() else None
    irisd = float(np.std(iri, ddof=1)) if len(iri) > 1 and np.isfinite(iri).all() else None
    return {
        'km_inicial': round(km_a, 3), 'km_final': round(km_b, 3),
        'comprimento_km': round(km_b - km_a, 3), 'n_intervalos': int(hi - lo),
        'd0_medio': round(d0m, 2) if d0m is not None and np.isfinite(d0m) else None,
        'd0_desvio': round(d0sd, 2) if d0sd is not None else None,
        'd0_erro_padrao': round(d0sd / np.sqrt(hi - lo), 2) if d0sd is not None else None,   # B&K Tab. 2
        'd0_cv': round(d0cv, 3) if d0cv is not None and np.isfinite(d0cv) else None,
        'd0_frac_medido': round(float(a['d0_obs'][lo:hi].mean()), 3),
        'igg_medio': round(iggm, 2) if iggm is not None and np.isfinite(iggm) else None,
        'igg_desvio': round(iggsd, 2) if iggsd is not None else None,
        'igg_cv': round(iggcv, 3) if iggcv is not None and np.isfinite(iggcv) else None,
        'iri_medio': round(irim, 2) if irim is not None and np.isfinite(irim) else None,
        'iri_desvio': round(irisd, 2) if irisd is not None else None,
        'iri_cv': round(iricv, 3) if iricv is not None and np.isfinite(iricv) else None,
        'coerente': coerente,
    }

def montar_segmentos(tipo, sre, inicios, a, km_min, km_max):
    N = len(a['ini'])
    idx = [0] + sorted(inicios) + [N]
    kms = [km_min] + [float(a['ini'][s]) for s in sorted(inicios)] + [km_max]
    linhas = []
    for i in range(len(idx) - 1):
        row = {'tipo': tipo, 'sre': sre, 'seg_id': i + 1}
        row.update(atributos(a, idx[i], idx[i + 1], kms[i], kms[i + 1]))
        linhas.append(row)
    return linhas

def segmentar_sre(sre, a, alpha_func, beta, origem_beta, cam, cotovelo):
    N = len(a['ini'])
    ini = a['ini']
    km_min, km_max = float(ini.min()), float(a['fim'].max())
    tem_igg = bool(np.isfinite(a['igg']).all())
    tem_d0 = cam is not None
    vf = piso_variancia(a['d0'])

    # Funcional (v6): PELT l2 sobre IGG
    func_s, pen_func, _ = segmentar_funcional_sre(a, alpha_func) if tem_igg else ([], None, None)
    n_func_bruto = len(func_s)
    if FUNDIR_FUNC and tem_igg:
        func_s = fundir_testes(func_s, [(a['igg'], False)], N)

    # Estrutural (B&K): solucao do caminho CROPS no beta escolhido
    if tem_d0:
        K = int(k_otimo(cam['Ks'], cam['Q'], [beta])[0])
        estrut_s = [int(b) for b in cam['bkps'][K][:-1]]
    else:
        estrut_s = []
    n_estrut_bruto = len(estrut_s)
    if FUNDIR_ESTRUT and tem_d0:
        estrut_s = fundir_testes(estrut_s, [(a['d0'], True)], N)

    # Combinada (2 etapas): bases funcionais -> PELT normal (mesmo beta) dentro de cada base
    if tem_d0 and SUBSEGMENTAR_COMB:
        lims = [0] + func_s + [N]
        sub_s = []
        for i in range(len(lims) - 1):
            sub_s.extend(segmentar_estrutural_trecho(a['d0'], lims[i], lims[i + 1], beta, vf))
    else:
        sub_s = estrut_s
    func_km = [float(ini[s]) for s in func_s]
    comb_km = reconciliar_combinada(func_km, [float(ini[s]) for s in sub_s], km_min, km_max)
    comb_s = [int(i) for i in np.searchsorted(ini, comb_km)]
    n_comb_bruto = len(comb_s)
    if FUNDIR_COMB:
        series = ([(a['d0'], True)] if tem_d0 else []) + ([(a['igg'], False)] if tem_igg else [])
        if series:
            comb_s = fundir_testes(comb_s, series, N)

    # Auditoria de residuos da segmentacao estrutural (B&K, "Test of Assumptions")
    if tem_d0:
        aud, z = auditar_residuos(a['d0'], a['d0_obs'], estrut_s)
    else:
        aud, z = {'n_residuos': 0, 'n_segmentos': 1}, None
    aud = {'sre': sre, 'beta': round(float(beta), 4) if tem_d0 else None, **aud}

    linhas = (montar_segmentos('funcional', sre, func_s, a, km_min, km_max)
              + montar_segmentos('estrutural', sre, estrut_s, a, km_min, km_max)
              + montar_segmentos('combinada', sre, comb_s, a, km_min, km_max))
    info = {'sre': sre, 'n_intervalos': N, 'extensao_km': round(km_max - km_min, 3),
            'frac_d0_medido': round(float(a['d0_obs'].mean()), 3),
            'pen_func': pen_func,
            'beta_estrut': round(float(beta), 4) if tem_d0 else None, 'origem_beta': origem_beta if tem_d0 else 'sem_d0',
            'beta_cotovelo_sre': round(cotovelo[1], 4) if cotovelo else None,
            'K_cotovelo_sre': cotovelo[0] if cotovelo else None,
            'beta_zero_quebras': round(float(cam['sup'][1]), 4) if tem_d0 and len(cam['Ks']) > 1 else None,
            'n_solucoes_crops': len(cam['Ks']) if tem_d0 else 0,
            'piso_var_d0': round(vf, 4) if tem_d0 else None,
            'quebras_func_pelt': n_func_bruto, 'quebras_func': len(func_s),
            'quebras_estrut_pelt': n_estrut_bruto, 'quebras_estrut': len(estrut_s),
            'quebras_comb_pelt': n_comb_bruto, 'quebras_comb': len(comb_s)}
    plot = {'func_s': func_s, 'estrut_s': estrut_s, 'comb_s': comb_s,
            'func_km': func_km, 'estrut_km': [float(ini[s]) for s in estrut_s],
            'comb_km': [float(ini[s]) for s in comb_s], 'beta': float(beta) if tem_d0 else None,
            'cotovelo': cotovelo, 'z': z, 'aud': aud,
            'cam': {k: cam[k] for k in ('Ks', 'Q', 'inf', 'sup')} if tem_d0 else None}
    return linhas, info, plot

t0 = time.time()
resultados = Parallel(n_jobs=N_JOBS, batch_size=16)(
    delayed(segmentar_sre)(s, grupos[s], ALPHA_KM_FUNC_OPT, BETA_SRE.get(s, BETA_REDE), ORIGEM_BETA.get(s, 'rede'),
                           CAMINHOS.get(s), COTOVELO.get(s)) for s in sres)

df_segmentos = pd.DataFrame([l for r in resultados for l in r[0]])
df_info = pd.DataFrame([r[1] for r in resultados])
plots = {r[1]['sre']: r[2] for r in resultados}
TEMPOS['segmentacao'] = time.time() - t0

n_tipo = df_segmentos['tipo'].value_counts()
print(f"OK Malha processada em {TEMPOS['segmentacao']:.1f}s — {len(df_segmentos)} segmentos totais "
      f"(func={n_tipo.get('funcional', 0)} | estrut={n_tipo.get('estrutural', 0)} | comb={n_tipo.get('combinada', 0)})")
for t, flag in [('func', FUNDIR_FUNC), ('estrut', FUNDIR_ESTRUT), ('comb', FUNDIR_COMB)]:
    if flag:
        print(f"  fusao ({t}): {df_info[f'quebras_{t}_pelt'].sum()} -> {df_info[f'quebras_{t}'].sum()} quebras")
_est = df_segmentos[(df_segmentos['tipo'] == 'estrutural') & df_segmentos['sre'].isin(CAMINHOS)]
print(f"  estrutural (SREs com d0): {len(_est)} segmentos | comprimento mediano {_est['comprimento_km'].median():.2f} km | "
      f"p10-p90 {_est['comprimento_km'].quantile(0.1):.2f}-{_est['comprimento_km'].quantile(0.9):.2f} km")

In [ ]:
# ─── 7. Auditoria dos Residuos na Malha (Biswas & Kuna, "Test of Assumptions") ─
df_auditoria = pd.DataFrame([plots[s]['aud'] for s in sres])
aud_val = df_auditoria.dropna(subset=['shapiro_p']).copy()

# residuos padronizados empilhados na malha
z_rede = np.concatenate([plots[s]['z'][np.isfinite(plots[s]['z'])] for s in sres if plots[s]['z'] is not None])
ks_rede = stats.kstest(z_rede, 'norm') if len(z_rede) else None
nt_rede = stats.normaltest(z_rede) if len(z_rede) >= 20 else None

resumo_aud = pd.DataFrame([
    {'metrica': 'SREs auditados (>= N_MIN_AUDITORIA d0 medidos)', 'valor': int(len(aud_val))},
    {'metrica': '% normalidade nao rejeitada (Shapiro-Wilk e KS, residuo padronizado)',
     'valor': round(100 * float(aud_val['normalidade_ok'].mean()), 1) if len(aud_val) else None},
    {'metrica': '% Shapiro-Wilk p > alpha (residuo padronizado)',
     'valor': round(100 * float((aud_val['shapiro_p'] > ALPHA_TESTE).mean()), 1) if len(aud_val) else None},
    {'metrica': '% KS p > alpha (residuo padronizado)',
     'valor': round(100 * float((aud_val['ks_p'] > ALPHA_TESTE).mean()), 1) if len(aud_val) else None},
    {'metrica': '% Shapiro-Wilk p > alpha (residuo bruto, como no artigo)',
     'valor': round(100 * float((aud_val['shapiro_p_res_bruto'] > ALPHA_TESTE).mean()), 1) if len(aud_val) else None},
    {'metrica': '% independencia nao rejeitada (Ljung-Box)',
     'valor': round(100 * float(aud_val['independencia_ok'].mean()), 1) if len(aud_val) else None},
    {'metrica': 'ACF lag 1 mediana', 'valor': round(float(aud_val['acf_lag1'].median()), 4) if len(aud_val) else None},
    {'metrica': '% heterocedasticos entre segmentos (Brown-Forsythe) -> justificam custo normal',
     'valor': round(100 * float(aud_val['heterocedastico'].dropna().mean()), 1) if len(aud_val) else None},
    {'metrica': 'razao desvio max/min mediana',
     'valor': round(float(aud_val['razao_desvio_max_min'].dropna().median()), 3) if len(aud_val) else None},
    {'metrica': 'erro padrao medio de d0 por secao (mediana)',
     'valor': round(float(aud_val['erro_padrao_medio'].dropna().median()), 2) if len(aud_val) else None},
    {'metrica': 'REDE: n residuos padronizados', 'valor': int(len(z_rede))},
    {'metrica': "REDE: D'Agostino K2 / p",
     'valor': f"{nt_rede.statistic:.2f} / {nt_rede.pvalue:.3e}" if nt_rede else None},
    {'metrica': 'REDE: KS D / p',
     'valor': f"{ks_rede.statistic:.4f} / {ks_rede.pvalue:.3e}" if ks_rede else None},
    {'metrica': 'REDE: assimetria / curtose excesso',
     'valor': f"{stats.skew(z_rede):.3f} / {stats.kurtosis(z_rede):.3f}" if len(z_rede) else None},
])
for _, r in resumo_aud.iterrows():
    print(f"  {r['metrica']}: {r['valor']}")

# ── Figura de auditoria (B&K Figs. 11-13) ──
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
ax = axes[0, 0]
if len(z_rede):
    am = np.random.default_rng(0).choice(z_rede, size=min(4000, len(z_rede)), replace=False)
    (osm, osr), (sl, ic, _) = stats.probplot(am, dist='norm')
    ax.plot(osm, osr, 'o', ms=2, alpha=0.5, color='#1F4E79')
    ax.plot(osm, sl * np.asarray(osm) + ic, color='#C65A57', lw=1.2)
ax.set_xlabel('quantis teoricos N(0,1)'); ax.set_ylabel('residuos padronizados z')
ax.set_title('(a) Q-Q dos residuos padronizados na malha (B&K Fig. 11)')

ax = axes[0, 1]
if len(aud_val):
    ax.hist(aud_val['shapiro_p'].dropna(), bins=25, alpha=0.75, color='#1F4E79', label='z padronizado (modelo normal)')
    ax.hist(aud_val['shapiro_p_res_bruto'].dropna(), bins=25, alpha=0.55, color='#B8860B', label='r bruto (variancia comum)')
    ax.axvline(ALPHA_TESTE, color='#C65A57', ls='--', lw=1.2, label=f'alpha={ALPHA_TESTE}')
    ax.legend(fontsize=8)
ax.set_xlabel('p-valor Shapiro-Wilk por SRE'); ax.set_ylabel('n. de SREs')
ax.set_title('(b) Normalidade por SRE (B&K Tab. 1)')

if ref is not None:
    a_ref = grupos[ref]
    r_ref, _, _, _ = residuos_segmentos(a_ref['d0'], a_ref['d0_obs'], plots[ref]['estrut_s'])
    rr = r_ref[np.isfinite(r_ref)]
    ax = axes[1, 0]
    if len(rr) >= 4:
        h = int(max(1, min(N_LAGS_ACF, len(rr) // 4)))
        rc = rr - rr.mean(); den = float((rc ** 2).sum())
        acf = [1.0] + [float((rc[:-k] * rc[k:]).sum() / den) if den > 0 else 0.0 for k in range(1, h + 1)]
        lags = np.arange(len(acf))
        ax.vlines(lags, 0, acf, color='#1F4E79', lw=1.5); ax.axhline(0, color='black', lw=0.7)
        b = 1.96 / np.sqrt(len(rr))
        ax.axhline(b, color='#1F4E79', ls='--'); ax.axhline(-b, color='#1F4E79', ls='--')
    ax.set_xlabel('lag'); ax.set_ylabel('ACF')
    ax.set_title(f'(c) ACF dos residuos do SRE {ref} (B&K Fig. 12)')

    ax = axes[1, 1]
    m = np.isfinite(r_ref)
    ax.plot(a_ref['centro'][m], r_ref[m], 'o', ms=3, color='#1F4E79', alpha=0.8)
    ax.axhline(0, color='#C65A57', lw=1.0)
    for f in plots[ref]['estrut_km']:
        ax.axvline(f, color='#1F4E79', ls=':', lw=1.0, alpha=0.8)
    ax.set_xlabel('km'); ax.set_ylabel('residuo de d0 (um)')
    ax.set_title(f'(d) Residuos vs km do SRE {ref} (B&K Fig. 13)')

for ax in axes.ravel():
    ax.grid(True, ls=':', alpha=0.5)
plt.tight_layout()
AUD_PNG = os.path.join(OUTPUT_DIR, '_auditoria_residuos.png')
plt.savefig(AUD_PNG, dpi=DPI, bbox_inches='tight'); plt.show()

In [ ]:
# ─── 8. Geracao dos Graficos por SRE (paralelo, backend Agg) ──────────────────
def plot_sre(sre, a, p, output_dir, dpi, beta_min):
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg

    km_c = a['centro']; d0 = a['d0']; aca = a['aca']; igg = a['igg']; iri = a['iri']
    func_km = p['func_km']; estrut_km = p['estrut_km']; comb_km = p['comb_km']

    fig = Figure(figsize=(14, 9.5))
    FigureCanvasAgg(fig)
    gs = fig.add_gridspec(4, 2, height_ratios=[1.15, 0.85, 0.85, 1.0], hspace=0.30, wspace=0.22)
    ax_d0  = fig.add_subplot(gs[0, :])
    ax_aca = fig.add_subplot(gs[1, :], sharex=ax_d0)
    ax_igg = fig.add_subplot(gs[2, :], sharex=ax_d0)
    ax_elb = fig.add_subplot(gs[3, 0])
    ax_qq  = fig.add_subplot(gs[3, 1])

    # (1) d0 bruto + media e +-1 desvio por secao estrutural (B&K Fig. 10)
    ax_d0.plot(km_c, d0, color='#333333', lw=0.8, alpha=0.75, label='d0')
    lim = [0] + p['estrut_s'] + [len(d0)]
    for i in range(len(lim) - 1):
        lo, hi = lim[i], lim[i + 1]
        seg = d0[lo:hi]
        if len(seg) and np.isfinite(seg).all():
            m = float(seg.mean()); sd = float(seg.std(ddof=1)) if len(seg) > 1 else 0.0
            x0 = float(a['ini'][lo]); x1 = float(a['ini'][hi]) if hi < len(d0) else float(a['fim'][-1])
            ax_d0.hlines(m, x0, x1, color='#C65A57', lw=2.0)
            ax_d0.fill_between([x0, x1], m - sd, m + sd, color='#C65A57', alpha=0.15)
    ax_d0.set_ylabel('d0 (um)', color='#333333')
    b_txt = f"{p['beta']:.2f}" if p['beta'] is not None else 'n/a'
    ax_d0.set_title(f"SRE {sre} | Func={len(func_km)+1} | Estrut(B&K normal, beta={b_txt})={len(estrut_km)+1} | "
                    f"Comb={len(comb_km)+1}")

    # (2) ACA apenas como referencia visual do CDA
    ax_aca.plot(km_c, aca, color='#B8860B', lw=1.0)
    ax_aca.set_ylabel('aca (ref. CDA)', color='#B8860B')

    # (3) IGG + IRI
    ax_igg.plot(km_c, igg, color='#C65A57', lw=0.95, label='IGG')
    ax_igg.set_ylabel('IGG', color='#C65A57'); ax_igg.set_xlabel('Distancia Linear (km)')
    ax_iri = ax_igg.twinx()
    ax_iri.plot(km_c, iri, color='#2E7D32', lw=0.8, alpha=0.65, label='IRI')
    ax_iri.set_ylabel('IRI (m/km)', color='#2E7D32')

    for ax in (ax_d0, ax_aca, ax_igg):
        ax.grid(True, linestyle=':', alpha=0.5)
        for f in func_km:
            ax.axvline(f, color='#555555', ls='--', lw=0.9, alpha=0.8)
        for f in estrut_km:
            ax.axvline(f, color='#1F4E79', ls=':', lw=1.2, alpha=0.85)
        for f in comb_km:
            ax.axvline(f, color='#C65A57', ls='-', lw=1.1, alpha=0.75)

    handles = [
        Line2D([0], [0], color='#555555', lw=1.0, ls='--', label='Funcional (IGG l2)'),
        Line2D([0], [0], color='#1F4E79', lw=1.2, ls=':',  label='Estrutural (d0 PELT normal)'),
        Line2D([0], [0], color='#C65A57', lw=1.1, ls='-',  label='Combinada (2 etapas)'),
    ]
    ax_d0.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 1.0), ncol=3,
                 fontsize=8, frameon=False)   # fora da area do grafico
    ax_d0.set_title(ax_d0.get_title(), pad=24)

    # (4a) Diagnostico do SRE: beta x n. de quebras (B&K Fig. 9)
    cam = p['cam']
    if cam is not None and len(cam['Ks']) > 1:
        g = np.linspace(beta_min, max(cam['sup'][1] * 1.1, beta_min + 1), 200)
        ax_elb.step(g, k_otimo(cam['Ks'], cam['Q'], g), where='post', color='#1F4E79', lw=1.2)
        ax_elb.plot(cam['inf'][1:], cam['Ks'][1:], 'o', color='#1F4E79', ms=3)
        ax_elb.axvline(p['beta'], color='#C65A57', lw=1.2, label=f"beta usado: {p['beta']:.2f}")
        if p['cotovelo'] is not None:
            ax_elb.axvline(p['cotovelo'][1], color='#555555', ls='--', lw=1.0,
                           label=f"cotovelo SRE: {p['cotovelo'][1]:.2f} (K={p['cotovelo'][0]})")
        ax_elb.legend(fontsize=7.5)
    ax_elb.set_xlabel('penalidade beta'); ax_elb.set_ylabel('n. de quebras')
    ax_elb.set_title('Diagnostico CROPS (B&K Fig. 9)', fontsize=9); ax_elb.grid(True, ls=':', alpha=0.5)

    # (4b) Q-Q dos residuos padronizados (B&K Fig. 11)
    zz = p['z'][np.isfinite(p['z'])] if p['z'] is not None else []
    if len(zz) >= 3 and np.ptp(zz) > 0:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            (osm, osr), (sl, ic, _) = stats.probplot(zz, dist='norm')
        ax_qq.plot(osm, osr, 'o', ms=3, color='#1F4E79', alpha=0.75)
        ax_qq.plot(osm, sl * np.asarray(osm) + ic, color='#C65A57', lw=1.1)
    sw_p = p['aud'].get('shapiro_p'); lb_p = p['aud'].get('ljung_box_p')
    ax_qq.set_title(f'Q-Q residuos (Shapiro p={sw_p}, Ljung-Box p={lb_p})', fontsize=9)
    ax_qq.set_xlabel('quantis N(0,1)'); ax_qq.set_ylabel('z'); ax_qq.grid(True, ls=':', alpha=0.5)

    png_path = os.path.join(output_dir, f'{sre}.png')
    fig.savefig(png_path, dpi=dpi, bbox_inches='tight')
    return png_path

t0 = time.time()
sres_plot = [s for s in sres if len(grupos[s]['ini']) >= N_MIN_GRAFICO]
if GERAR_GRAFICOS:
    arquivos_png = Parallel(n_jobs=N_JOBS, batch_size=8)(
        delayed(plot_sre)(s, grupos[s], plots[s], OUTPUT_DIR, DPI, BETA_MIN_EF) for s in sres_plot)
else:
    arquivos_png = []
for p in (CALIB_PNG, AUD_PNG):
    if os.path.exists(p):
        arquivos_png.append(p)
TEMPOS['graficos'] = time.time() - t0
print(f"Graficos: {len(arquivos_png)} gerados ({len(sres) - len(sres_plot)} SREs com < {N_MIN_GRAFICO} intervalos omitidos) "
      f"em {TEMPOS['graficos']:.1f}s")

In [ ]:
# ─── 9. Exportacao da Planilha e ZIP dos Graficos ─────────────────────────────
t0 = time.time()
params_df = pd.DataFrame([
    {'parametro': 'INPUT_FILE', 'valor': INPUT_FILE},
    {'parametro': 'MODEL_FUNC', 'valor': MODEL_FUNC},
    {'parametro': 'MODEL_ESTRUT', 'valor': f"{MODEL_ESTRUT} sobre d0 bruto (Biswas & Kuna, 2022)"},
    {'parametro': 'MODO_PENALIDADE', 'valor': MODO_PENALIDADE},
    {'parametro': 'BETA_REDE', 'valor': round(BETA_REDE, 4)},
    {'parametro': 'BETA_COTOVELO_REDE', 'valor': round(BETA_COTOVELO_REDE, 4)},
    {'parametro': 'BETA_BIC_3lnN', 'valor': round(BETA_BIC, 4)},
    {'parametro': 'BETA_FIXO', 'valor': BETA_FIXO},
    {'parametro': 'BETA_MIN_EF', 'valor': round(BETA_MIN_EF, 4)},
    {'parametro': 'BETA_MAX_REDE', 'valor': round(BETA_MAX_REDE, 4)},
    {'parametro': 'PELT_EXATO', 'valor': PELT_EXATO},
    {'parametro': 'VAR_FLOOR_FRAC', 'valor': VAR_FLOOR_FRAC},
    {'parametro': 'MIN_SIZE', 'valor': MIN_SIZE},
    {'parametro': 'JUMP', 'valor': JUMP},
    {'parametro': 'ALPHA_KM_FUNC_OPT', 'valor': round(ALPHA_KM_FUNC_OPT, 6)},
    {'parametro': 'CRITERIO_ALPHA_FUNC', 'valor': CRITERIO_ALPHA_FUNC},
    {'parametro': 'MARGEM_PEN', 'valor': MARGEM_PEN},
    {'parametro': 'N_CAND_PEN', 'valor': N_CAND_PEN},
    {'parametro': 'FUNDIR_FUNC / ESTRUT / COMB', 'valor': f'{FUNDIR_FUNC} / {FUNDIR_ESTRUT} / {FUNDIR_COMB}'},
    {'parametro': 'ALPHA_TESTE', 'valor': ALPHA_TESTE},
    {'parametro': 'SUBSEGMENTAR_COMB', 'valor': SUBSEGMENTAR_COMB},
    {'parametro': 'BUFFER_KM', 'valor': BUFFER_KM},
    {'parametro': 'MIN_COMP_COMB_KM', 'valor': MIN_COMP_COMB_KM},
])

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as xw:
    for tipo in ['funcional', 'estrutural', 'combinada']:
        sub = df_segmentos[df_segmentos['tipo'] == tipo]
        if not sub.empty:
            sub.drop(columns=['tipo']).to_excel(xw, sheet_name=tipo.upper(), index=False)

    resumo = df_segmentos.groupby(['sre', 'tipo']).agg(
        n_segmentos=('seg_id', 'count'),
        comp_total_km=('comprimento_km', 'sum'),
        comp_medio_km=('comprimento_km', 'mean'),
        coerentes=('coerente', 'sum'),
        d0_desvio_medio=('d0_desvio', 'mean'),
        d0_erro_padrao_medio=('d0_erro_padrao', 'mean'),
        d0_cv_medio=('d0_cv', 'mean'),
        igg_desvio_medio=('igg_desvio', 'mean'),
        igg_cv_medio=('igg_cv', 'mean'),
        iri_desvio_medio=('iri_desvio', 'mean'),
        iri_cv_medio=('iri_cv', 'mean'),
    ).round(4).reset_index()
    resumo.to_excel(xw, sheet_name='RESUMO', index=False)
    df_info.to_excel(xw, sheet_name='PENALIDADES', index=False)
    df_auditoria.to_excel(xw, sheet_name='AUDITORIA', index=False)
    resumo_aud.to_excel(xw, sheet_name='AUDITORIA_RESUMO', index=False)
    DIAG_REDE.to_excel(xw, sheet_name='DIAGNOSTICO_REDE', index=False)
    TAB_CROPS.to_excel(xw, sheet_name='CROPS', index=False)
    if CURVA_FUNC is not None:
        CURVA_FUNC.to_excel(xw, sheet_name='CALIBRACAO_FUNC', index=False)
    params_df.to_excel(xw, sheet_name='PARAMETROS', index=False)

with zipfile.ZipFile(ZIP_FILE, 'w', zipfile.ZIP_STORED) as zf:   # PNGs ja sao comprimidos
    for p in arquivos_png:
        zf.write(p, arcname=os.path.join('graficos', os.path.basename(p)))
    if os.path.exists(OUTPUT_FILE):
        zf.write(OUTPUT_FILE, arcname=os.path.basename(OUTPUT_FILE))
TEMPOS['exportacao'] = time.time() - t0

print(f"OK Planilha: {OUTPUT_FILE}")
print(f"OK ZIP gerado: {ZIP_FILE} ({len(arquivos_png)} graficos + planilha) em {TEMPOS['exportacao']:.1f}s")
print("Tempos (s):", {k: round(v, 1) for k, v in TEMPOS.items()}, f"| total={sum(TEMPOS.values()):.1f}s")

if IN_COLAB:
    files.download(ZIP_FILE)
else:
    print(f"Salvo em: {os.path.abspath(ZIP_FILE)}")